# FCSG-Net training run

Run `checks.ipynb` first. This notebook only trains.

Sessions die at 12 hours and the weekly quota is 30 GPU-hours, so plan on three
or four sessions rather than one heroic run. Checkpoints land in
`/kaggle/working/ckpt`, which persists as the notebook output.

In [ ]:
# Clone once, pull on every later run. Editing code here is a trap: the next
# session starts from a fresh container and your edits are gone. Edit locally,
# push, re-run this cell.
import os, subprocess, sys
REPO_URL = "https://github.com/whynotramaa/fcsg-capstone"
REPO = "/kaggle/working/fcsg-capstone"
if os.path.isdir(REPO):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Attach a public DIV2K dataset in the sidebar (Add Input -> search "DIV2K"),
# then run this. Nothing hardcodes a slug, so any DIV2K dataset layout works as
# long as it contains DIV2K_train_HR and DIV2K_valid_HR directories.
sys.path[:0] = ["/kaggle/working/fcsg-capstone", "/kaggle/working/fcsg-capstone/src"]
from fcsg_net.utils import resolve_div2k
DATA = "/kaggle/input"
train_dir, val_dir = resolve_div2k(DATA)
print("train:", train_dir)
print("valid:", val_dir)

## Resume from a previous session

Skip on the first run. Afterwards: attach this notebook's previous output as an
input dataset in the sidebar, then run this cell to seed the checkpoint
directory from it.

In [ ]:
import glob, shutil, os
prev = sorted(glob.glob("/kaggle/input/*/ckpt/ckpt_*.pt")) or sorted(glob.glob("/kaggle/input/*/ckpt_*.pt"))
os.makedirs("/kaggle/working/ckpt", exist_ok=True)
if prev:
    for src in [prev[-1]]:  # only the newest; train.py resumes from the highest step
        shutil.copy(src, "/kaggle/working/ckpt/")
        print("seeded", src)
    for log in glob.glob("/kaggle/input/*/ckpt/train_log.csv"):
        shutil.copy(log, "/kaggle/working/ckpt/")
        print("seeded", log)
else:
    print("no previous checkpoint attached, starting fresh")

## Train

Watch the first few lines: it must print the device, the parameter count, and
either `RESUMED from ...` or `starting from step 0`. Then leave it alone.

In [ ]:
!python training/train.py --config configs/dncnn.toml --data /kaggle/input --out /kaggle/working/ckpt

## Results

Re-run these against the finished checkpoint.

In [ ]:
import glob
ckpt = sorted(glob.glob("/kaggle/working/ckpt/ckpt_*.pt"))[-1]
!python evaluation/eval.py --ckpt $ckpt --data /kaggle/input --out /kaggle/working/results \
    --limit 20 --notes final
!python evaluation/plots.py --csv /kaggle/working/ckpt/train_log.csv \
    --out /kaggle/working/results/figures

In [ ]:
from IPython.display import Image as Show, display
import pandas as pd
display(pd.read_csv("/kaggle/working/results/benchmark.csv"))
for f in ["loss_curve.png", "psnr_curve.png", "qualitative.png"]:
    display(Show(f"/kaggle/working/results/figures/{f}"))

Phase 1 exit test (plan.md): DnCNN at roughly 29 dB on DIV2K
validation at sigma=25. Download `/kaggle/working/results/` and commit it before
closing the session.